# Backend Architecture

FastAPI application structure for the NovaCRM KPI backend.

## 1. App Factory

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(
    title="Executive KPI Report API",
    version="1.0.0",
    description="Backend API for NovaCRM Executive KPI Dashboard and PDF report generation.",
)

# Explicit origins instead of wildcard: wildcard blocks credentialed requests
# and makes it impossible to use cookies or Authorization headers in the future.
app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:3000",
        "http://localhost:3052",
        "http://localhost:3053",
        "https://*.vercel.app",
    ],
    allow_methods=["GET"],
    allow_headers=["*"],
)

## 2. Data Loader

Module-level singleton load: parquet files are read once at import time and held in memory for the process lifetime.

In [ ]:
import os
from typing import List, Optional

import pandas as pd

# Resolve data directory relative to this module's location
DATA_DIR = os.path.join(
    os.path.dirname(os.path.abspath("backend/kpi_backend/data_loader.py")),
    "backend", "kpi_backend", "..", "..", "data", "processed"
)

_PARQUETS = {
    "monthly_metrics": "monthly_metrics.parquet",
    "segment_metrics": "segment_metrics.parquet",
    "monthly_kpis": "monthly_kpis.parquet",
    "segment_kpis": "segment_kpis.parquet",
}


def _load(name: str) -> pd.DataFrame:
    path = os.path.join(DATA_DIR, _PARQUETS[name])
    try:
        df = pd.read_parquet(path)
        if "month" in df.columns:
            df["month"] = pd.to_datetime(df["month"])
        print(f"  Loaded {name}: {len(df):,} rows")
        return df
    except FileNotFoundError:
        print(f"  WARNING: {path} not found -- using empty DataFrame")
        return pd.DataFrame()

In [ ]:
# Parquet over SQLite: columnar format gives ~3x faster reads for analytical
# queries that access a subset of columns; schema is enforced at write time;
# no connection management required.

DATA_PATH = "data/processed"

monthly_metrics = pd.read_parquet(f"{DATA_PATH}/monthly_metrics.parquet")
monthly_metrics["month"] = pd.to_datetime(monthly_metrics["month"])

segment_metrics = pd.read_parquet(f"{DATA_PATH}/segment_metrics.parquet")
segment_metrics["month"] = pd.to_datetime(segment_metrics["month"])

monthly_kpis = pd.read_parquet(f"{DATA_PATH}/monthly_kpis.parquet")
monthly_kpis["month"] = pd.to_datetime(monthly_kpis["month"])

segment_kpis = pd.read_parquet(f"{DATA_PATH}/segment_kpis.parquet")
segment_kpis["month"] = pd.to_datetime(segment_kpis["month"])

print(f"monthly_metrics: {len(monthly_metrics)} rows, {monthly_metrics.shape[1]} cols")
print(f"monthly_kpis:    {len(monthly_kpis)} rows, {monthly_kpis.shape[1]} cols")
print(f"segment_metrics: {len(segment_metrics)} rows")
print(f"segment_kpis:    {len(segment_kpis)} rows")

## 3. Filter Chain

`apply_filters` returns a filtered copy, preserving the original singleton for subsequent requests.

In [ ]:
def apply_filters(
    df: pd.DataFrame,
    segment: Optional[str] = None,
    start_month: Optional[str] = None,
    end_month: Optional[str] = None,
) -> pd.DataFrame:
    """Filter a DataFrame by segment and/or month range."""
    filtered = df.copy()

    if segment and "segment" in filtered.columns:
        filtered = filtered[filtered["segment"] == segment]

    if "month" in filtered.columns:
        if start_month:
            filtered = filtered[filtered["month"] >= pd.Timestamp(start_month)]
        if end_month:
            # Include the full end month: Timestamp("2024-12") = 2024-12-01, so <= captures all Dec rows
            filtered = filtered[filtered["month"] <= pd.Timestamp(end_month)]

    return filtered


# Smoke test: filter to last 6 months of the dataset
last_month = monthly_metrics["month"].max().strftime("%Y-%m")
six_months_ago = (monthly_metrics["month"].max() - pd.DateOffset(months=5)).strftime("%Y-%m")

sample = apply_filters(monthly_metrics, start_month=six_months_ago, end_month=last_month)
print(f"Filtered: {len(sample)} rows ({six_months_ago} to {last_month})")
print(sample[["month", "mrr"]].to_string(index=False))

## 4. Router Mounting

Each domain (overview, revenue, customers, forecast, anomalies, report) lives in its own router module.

In [ ]:
# Illustrative -- actual imports require the kpi_backend package to be installed
# or the backend directory to be in PYTHONPATH.

router_config = [
    {"module": "kpi_backend.routers.overview",   "prefix": "/api/v1", "tag": "overview"},
    {"module": "kpi_backend.routers.revenue",    "prefix": "/api/v1", "tag": "revenue"},
    {"module": "kpi_backend.routers.customers",  "prefix": "/api/v1", "tag": "customers"},
    {"module": "kpi_backend.routers.forecast",   "prefix": "/api/v1", "tag": "forecast"},
    {"module": "kpi_backend.routers.anomalies",  "prefix": "/api/v1", "tag": "anomalies"},
    {"module": "kpi_backend.routers.report",     "prefix": "/api/v1", "tag": "report"},
]

# Single /api/v1 prefix on all routers: keeps route registration DRY.
# Each router uses a distinct path segment (/overview, /revenue, etc.)
# so there is no risk of endpoint collisions between modules.

for rc in router_config:
    print(f"  {rc['prefix']}/{rc['tag']}  <--  {rc['module']}.router")

## 5. Startup and Memory Model

Data is loaded at module import, not inside an `@app.on_event("startup")` handler, so it is available immediately when the module is first imported (including during tests).

In [ ]:
# Memory footprint estimate
import sys

dfs = {
    "monthly_metrics": monthly_metrics,
    "segment_metrics": segment_metrics,
    "monthly_kpis": monthly_kpis,
    "segment_kpis": segment_kpis,
}

for name, df in dfs.items():
    mem_kb = df.memory_usage(deep=True).sum() / 1024
    print(f"  {name:<20} {len(df):>4} rows  {mem_kb:>8.1f} KB")

# Total footprint is well under 10 MB -- safe to hold all four in process memory.
# At 24 months x 3 segments = 72 rows max; no streaming or lazy loading needed.

## 6. Dependency Pattern

Routers receive filtered DataFrames by calling `data_loader.apply_filters()` directly. FastAPI `Depends()` is not used because the filter parameters vary per endpoint and the data layer is stateless.

In [ ]:
# Pattern used in every router endpoint:

from typing import Optional

def example_endpoint(
    segment: Optional[str] = None,
    start_month: Optional[str] = None,
    end_month: Optional[str] = None,
):
    # Step 1: filter the singleton DataFrames
    kpis_df = apply_filters(monthly_kpis, segment=None, start_month=start_month, end_month=end_month)
    metrics_df = apply_filters(monthly_metrics, segment=None, start_month=start_month, end_month=end_month)

    # Step 2: guard against empty result
    if kpis_df.empty and metrics_df.empty:
        return {"error": "No data matches the selected filters."}

    # Step 3: use whichever DataFrame has data, sorted chronologically
    source = kpis_df if not kpis_df.empty else metrics_df
    source = source.sort_values("month") if "month" in source.columns else source

    last_row = source.iloc[-1]
    return {"last_month": str(last_row["month"]), "rows_in_period": len(source)}


# Demonstration
result = example_endpoint(start_month="2025-01", end_month="2025-06")
print(result)

## 7. Health Check Endpoint

In [ ]:
# The /health endpoint validates that all four datasets loaded successfully.
# It exposes row counts and available filter values for observability.

SEGMENTS = sorted(
    segment_metrics["segment"].dropna().unique().tolist()
) if "segment" in segment_metrics.columns else ["Starter", "Professional", "Enterprise"]

MONTHS = sorted(
    monthly_metrics["month"].dt.strftime("%Y-%m").unique().tolist()
) if "month" in monthly_metrics.columns else []

health_response = {
    "status": "ok",
    "datasets": {
        "monthly_metrics": len(monthly_metrics),
        "segment_metrics": len(segment_metrics),
        "monthly_kpis": len(monthly_kpis),
        "segment_kpis": len(segment_kpis),
    },
    "filters_available": {
        "segments": SEGMENTS,
        "months": len(MONTHS),
    },
}

import json
print(json.dumps(health_response, indent=2))